# Plots for Foundation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
os.chdir('../..')
sys.path.insert(1, os.path.join(sys.path[0], '../..'))

In [ ]:
import torch
import pandas as pd
from plotnine import *
import numpy as np
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

In [ ]:
MODELS_MHFA = {
    "Supervised (ground-truth labels)": "models/foundation/voxceleb2/sup_f-wavlm-base+_b-mhfa",
    "Supervised (pseudo-labels)": "models/foundation/voxceleb2/ssl_f-wavlm-base+_b-mhfa_km-50000_ahc-7500_dlg_ipl-3",
}

In [ ]:
MODELS_TDNN = {
    "Supervised (ground-truth labels) (frozen)":  "models/foundation/voxceleb2/wavlm-base+_frozen_tdnn_supervised",
    "Supervised (ground-truth labels)":           "models/foundation/voxceleb2/wavlm-base+_ft_tdnn_supervised",

    "Supervised (pseudo-labels) (frozen)":  "models/foundation/voxceleb2/wavlm-base+_frozen_tdnn_supervised-pseudo",
    "Supervised (pseudo-labels)":           "models/foundation/voxceleb2/wavlm-base+_ft_tdnn_supervised-pseudo",

    "SimCLR (frozen)":  "models/foundation/voxceleb2/wavlm-base+_frozen_tdnn_simclr",
    "SimCLR":           "models/foundation/voxceleb2/wavlm-base+_ft_tdnn_simclr",
}

## Load checkpoints

In [ ]:
def process_sslsv_key(k):
    parts = k.split(".")
    if parts[0] == "encoder":
        k = "frontend." + k[len("encoder."):]
    k = k.replace("frontend.pooling.", "backend.")
    k = k.replace("frontend.upstream.upstream.model.", "frontend.")
    return k

def process_wespeaker_key(k):
    parts = k.split(".")
    if parts[0] == "model":
        k = k[len("model."):]
    k = k.replace("frontend.upstream.upstream.model.", "frontend.")
    return k

def load_ckpt(model, models, epoch=None):
    model_path = models[model]
    is_wespeaker = model_path.split("/")[-1][:3] in ["sup", "ssl"]

    if epoch is not None:
        if is_wespeaker:
            ckpt_suffix = f"/models/model_{epoch + 1}.pt" if epoch < 10 else f"_ft/models/model_{epoch - 10 + 1}.pt"
        else:
            ckpt_suffix = f"/checkpoints/model_epoch-{epoch}.pt"
    else:
        if is_wespeaker:
            ckpt_suffix = "_ft/models/avg_model.pt"
        else:
            ckpt_suffix = "/checkpoints/model_avg.pt"

    ckpt = torch.load(f"{model_path}{ckpt_suffix}", map_location="cpu")
    if is_wespeaker:
        new_ckpt = {process_wespeaker_key(k):v for k, v in ckpt.items()}
    else:
        new_ckpt = {process_sslsv_key(k):v for k, v in ckpt["model"].items()}
    return new_ckpt

## Layer-wise weights

In [ ]:
def extract_mhfa_backend_weights(ckpt, model_name):
    data = []
    for key in ["backend.weights_k", "backend.weights_v"]:
        weights = torch.softmax(ckpt[key].float(), dim=-1).numpy()
        for layer, value in enumerate(weights):
            data.append({
                "model": model_name,
                "layer": layer,
                "weight": value,
                "type": "MHFA (Keys)" if key.endswith("weights_k") else "MHFA (Values)"
            })
    return pd.DataFrame(data)

def extract_tdnn_backend_weights(ckpt, model_name):
    data = []
    weights = torch.softmax(ckpt["frontend.layer_weights"].float(), dim=-1).numpy()
    for layer, value in enumerate(weights):
        data.append({
            "model": model_name,
            "layer": layer,
            "weight": value,
            "type": "TDNN"
        })
    return pd.DataFrame(data)

def plot_layer_weights(df):
    p = (
        ggplot(df, aes(x="layer", y="weight", color="model"))
        + geom_line(size=1)
        + geom_point(size=2)
        + facet_wrap("~type", ncol=1)
        + scale_x_continuous(breaks=[i for i in range(13)], labels=[str(i) for i in range(13)])
        + scale_y_continuous(limits=(0.0, 1.0))
        + scale_color_manual(values=MODELS_PALETTE)
        + labs(
            x="Layer",
            y="Weight",
            color="Model",
        )
        + theme_bw()
        + theme(
            figure_size=(10, 4.5),
            text=element_text(size=20),
            legend_title=element_blank(),
            legend_position="top",
            legend_key_spacing_x=20,
            axis_text_x=element_text(size=20),
            axis_text_y=element_text(size=20)
        )
    )
    return p

In [ ]:
from mizani.palettes import hue_pal
import matplotlib.pyplot as plt

MODELS = ["SimCLR", "Supervised (ground-truth labels)", "Supervised (pseudo-labels)"]

palette = hue_pal(h=0.01, l=0.6, s=0.65, color_space="hls")(len(MODELS))

plt.figure(figsize=(10, 1))
for i, color in enumerate(palette):
    plt.bar(i, 1, color=color)
plt.xticks(range(len(MODELS)), MODELS)
plt.show()

palette = dict(zip(MODELS, palette))

MODELS_PALETTE = palette

print(palette)

In [ ]:
df = pd.concat([
    extract_tdnn_backend_weights(load_ckpt("SimCLR", MODELS_TDNN), "SimCLR"),
    # extract_tdnn_backend_weights(load_ckpt("SimCLR (frozen)", MODELS_TDNN), "SimCLR (frozen)"),

    extract_tdnn_backend_weights(load_ckpt("Supervised (pseudo-labels)", MODELS_TDNN), "Supervised (pseudo-labels)"),
    # extract_tdnn_backend_weights(load_ckpt("Supervised (pseudo-labels) (frozen)", MODELS_TDNN), "Supervised (pseudo-labels) (frozen)"),

    extract_tdnn_backend_weights(load_ckpt("Supervised (ground-truth labels)", MODELS_TDNN), "Supervised (ground-truth labels)"),
    # extract_tdnn_backend_weights(load_ckpt("Supervised (frozen)", MODELS_TDNN), "Supervised (frozen)"),
])

p = plot_layer_weights(df)

p += scale_y_continuous(limits=(0.0, 0.3))
p += theme(figure_size=(10, 4.5), legend_position='none')

p.save("layer_weights_tdnn.pdf")

p

In [ ]:
df

In [ ]:
df = pd.concat([
    extract_mhfa_backend_weights(load_ckpt("Supervised (ground-truth labels)", MODELS_MHFA), "Supervised (ground-truth labels)"),
    extract_mhfa_backend_weights(load_ckpt("Supervised (pseudo-labels)", MODELS_MHFA), "Supervised (pseudo-labels)"),
])

p = plot_layer_weights(df)

p += scale_y_continuous(limits=(-0.01, 0.21), breaks=[.0, 0.1, 0.2])
p += theme(figure_size=(10, 4.5), legend_position='none')

p.save("layer_weights_mhfa.pdf")

p

In [ ]:
df

## Evolution of weights

In [ ]:
INIT_CKPT = torch.load("../WavLM-Base+.pt", map_location="cpu")["model"]

# CKPT_SUP = [load_ckpt("Supervised (ground-truth labels)", MODELS_TDNN, epoch=epoch) for epoch in range(19 + 1)]
CKPT_SSL = [load_ckpt("SimCLR", MODELS_TDNN, epoch=epoch) for epoch in range(19 + 1)]

In [ ]:
def compute_layerwise_distances(ckpts, init_ckpt, max_epoch=9):
    data = []

    keys = init_ckpt.keys()

    layer_prefixes = ["feature_extractor." if l == 0 else f"encoder.layers.{l - 1}." for l in range(13)]
    
    for epoch in range(max_epoch + 1):
        for layer_idx, prefix in enumerate(layer_prefixes):
            init_weights = [init_ckpt[k].flatten() for k in keys if k.startswith(prefix)]
            curr_weights = [
                ckpts[epoch]["frontend." + k].flatten() for k in keys
                if k.startswith(prefix)
            ]
            dist = 0
            if init_weights and curr_weights:
                init_vec = torch.cat(init_weights)
                curr_vec = torch.cat(curr_weights)
                dist = torch.norm(curr_vec - init_vec, p=2).item()
            data.append({
                "epoch": epoch + 1,
                "layer": layer_idx,
                "dist": dist
            })

    df = pd.DataFrame(data)

    df["dist_norm"] = df["dist"] / df["dist"].max()

    return df

In [ ]:
def plot_layerwise_distances(ckpts, init_ckpt):
    df = compute_layerwise_distances(ckpts, init_ckpt, max_epoch=19)
    
    p = (
        ggplot(df, aes(x="factor(epoch)", y="factor(layer)", fill="dist_norm"))
        + geom_tile(color=None, size=0, raster=True)
        + scale_fill_cmap(
            cmap_name="magma",
            breaks=[0.0, 0.5, 1.0],
            labels=["0.0", "0.5", "1.0"],
            limits=(0, 1))
        + scale_x_discrete(
            breaks=[i for i in [1, 5, 10, 15, 20]],
            labels=[str(i) for i in [10, 15, 20, 25, 30]]
        )
        + labs(x="Epoch", y="Layer", fill="Difference")
        + theme_bw()
        + theme(
            figure_size=(8, 4.75),
            text=element_text(size=18),
            panel_border=element_blank(),
            # legend_title=element_blank(),
            # legend_position="top",
            # legend_key_spacing_x=20
        )
    )

    return p, df

In [ ]:
p, df = plot_layerwise_distances(CKPT_SUP, INIT_CKPT)

p.save("layer_changes_sup.pdf")

p

In [ ]:
df.dist.min(), df.dist.max(), df.dist.mean(), df.dist.std()

In [ ]:
df[(df.epoch % 5 == 0) | (df.epoch == 1)][['epoch', 'layer', 'dist_norm']]

In [ ]:
p, df = plot_layerwise_distances(CKPT_SSL, INIT_CKPT)

p.save("layer_changes_ssl.pdf")

p

In [ ]:
df.dist.min(), df.dist.max(), df.dist.mean(), df.dist.std()

In [ ]:
df

In [ ]:
df[(df.epoch % 5 == 0) | (df.epoch == 1)][['epoch', 'layer', 'dist_norm']]

## Loss distributions

In [ ]:
def plot_loss_distributions(model, epochs):
    hist_list = []
    gmm_list = []
    thresh_list = []

    for epoch in epochs:
        if epoch <= 10:
            loss_vals_path = f"models/foundation/voxceleb2/{model}/loss_vals/epoch-{epoch}.txt"
        else:
            loss_vals_path = f"models/foundation/voxceleb2/{model}_ft/loss_vals/epoch-{epoch - 10}.txt"
        with open(loss_vals_path, "r") as f:
            losses = [float(line.strip().split()[0]) for line in f.readlines()]

        log_losses = np.log(losses)#[:100000]

        # Fit 2-component GMM
        gmm = GaussianMixture(
            n_components=2,
            random_state=0,
            covariance_type="full",
            tol=1e-5,
            max_iter=1000,
        )
        gmm.fit(log_losses.reshape(-1, 1))
        
        means = gmm.means_.flatten()
        covars = gmm.covariances_.flatten()
        weights = gmm.weights_.flatten()

        order = np.argsort(means)

        mean1, mean2 = means[order]
        covar1, covar2 = covars[order]
        weight1, weight2 = weights[order]

        x = np.linspace(min(log_losses), max(log_losses), 1000)
        g1 = weight1 * norm.pdf(x, mean1, np.sqrt(covar1))
        g2 = weight2 * norm.pdf(x, mean2, np.sqrt(covar2))

        # Find intersection
        intersection = np.argwhere(np.diff(np.sign(g1 - g2))).flatten()
        max1, max2 = x[np.argmax(g1)], x[np.argmax(g2)]
        good_intersection = x[intersection][
            (x[intersection] > min(max1, max2)) & (x[intersection] < max(max1, max2))
        ]
        assert len(good_intersection) == 1, "Wrong number of intersections"
        good_intersection = good_intersection[0]

        # Build dataframes
        df_hist = pd.DataFrame({"log_loss": log_losses, "epoch": epoch})
        df_gmm = (
            pd.DataFrame({"x": x, "g1": g1, "g2": g2})
            .melt(id_vars="x", var_name="component", value_name="density")
        )
        df_gmm["epoch"] = epoch
        df_gmm["component"] = pd.Categorical(
            df_gmm["component"], categories=["g1", "g2"], ordered=True
        )

        df_thresh = pd.DataFrame(
            {"xintercept": [good_intersection], "label": ["Threshold"], "epoch": [epoch]}
        )

        # Append to lists
        hist_list.append(df_hist)
        gmm_list.append(df_gmm)
        thresh_list.append(df_thresh)

    # Concatenate all epochs
    df_hist = pd.concat(hist_list, ignore_index=True)
    df_gmm = pd.concat(gmm_list, ignore_index=True)
    df_thresh = pd.concat(thresh_list, ignore_index=True)

    # Plot with facets
    p = (
        ggplot(df_hist, aes(x="log_loss"))
        + geom_histogram(aes(y="..density.."), bins=250, fill="gray", alpha=0.4)
        + geom_line(df_gmm, aes(x="x", y="density", color="component"), size=1)
        + geom_vline(df_thresh, aes(xintercept="xintercept", linetype="label"), color="#2b2b2b", size=1)
        + facet_wrap("~epoch", labeller=labeller(epoch=lambda e: f"Epoch {e}"), scales="free")
        + scale_color_manual(
            values={"g1": "#2EC267", "g2": "#F44336"},
            labels=["GMM Component #1 (Reliable)", "GMM Component #2 (Unreliable)"],
        )
        + scale_linetype_manual(values={"Threshold": "solid"}, labels=["Loss threshold"])
        + labs(
            x="log(loss)",
            y="Density",
            color="",
            linetype="",
        )
        + theme_bw()
        + theme(
            figure_size=(13, 4),
            text=element_text(size=16),
            legend_title=None,
            legend_position="top",
            legend_key_spacing_x=20,
            legend_box_spacing=0.02,
            legend_key=element_blank()
        )
    )

    return p

In [ ]:
p = plot_loss_distributions("sup_f-wavlm-base+_b-mhfa", epochs=[0, 10, 20])

p.save("loss_distributions_sup.pdf")

p

In [ ]:
p = plot_loss_distributions("ssl_f-wavlm-base+_b-mhfa_km-50000_ahc-7500", epochs=[0, 10, 20])

p.save("loss_distributions_ssl.pdf")

p

In [ ]:
p = plot_loss_distributions("ssl_f-wavlm-base+_b-mhfa_km-50000_ahc-7500_dlg", epochs=[0, 10, 20])

p.save("loss_distributions_ssl_dlg.pdf")

p